In [ ]:
import os

# 1. 强制清理旧残留
print("正在清理旧文件...")
!rm -rf Diffusion-Illusions
!rm -rf Diffusion-Illusion
!rm -rf master.zip

# 2. 克隆仓库
print("正在克隆仓库...")
!git clone https://github.com/RyannDaGreat/Diffusion-Illusions

# 3. 检查是否成功
if os.path.exists('Diffusion-Illusions'):
    print("✅ 仓库克隆成功！")
    
    # 4. 进入目录
    %cd Diffusion-Illusions
    
    # 5. 安装依赖
    print("正在安装依赖 (红色警告请忽略)...")
    !pip install -r requirements.txt
    !pip install mediapy easydict "numpy<2.0"
    
    print("\n✅✅ 环境初始化全部完成！")
    print("⚠️⚠️ 现在的关键步骤：请点击上方菜单 'Runtime' -> 'Restart session' 重启运行时！")
    
else:
    print("❌❌ 克隆还是失败了，请检查网络。")

In [ ]:
import os

# 定义仓库名字
repo_name = "Diffusion-Illusions"

# 检查当前是否已经在文件夹里了
if os.getcwd().endswith(repo_name):
    print(f"✅ 当前位置正确: {os.getcwd()}")
else:
    # 如果不在，就尝试进去
    if os.path.exists(repo_name):
        %cd {repo_name}
        print(f"✅ 已切换工作目录到: {os.getcwd()}")
    else:
        # 如果文件夹都不存在，说明之前的克隆没成功，重新克隆一下
        print("⚠️ 文件夹不存在，正在重新克隆...")
        !git clone https://github.com/RyannDaGreat/Diffusion-Illusions
        %cd {repo_name}
        print(f"✅ 克隆并切换完成: {os.getcwd()}")

In [ ]:
import os
import sys

# === 🚑 自动修复路径 (新增部分) ===
# 如果当前目录下没有 rp.py，但有一个 Diffusion-Illusions 文件夹，就进去
if not os.path.exists('rp.py') and os.path.exists('Diffusion-Illusions'):
    print("⚠️ 检测到目录位置不对，正在进入 Diffusion-Illusions 文件夹...")
    os.chdir('Diffusion-Illusions')

# 再次检查，确保 Python 能搜索到当前目录
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# === 下面是原本的代码 ===
from rp import *
import numpy as np
import rp
import torch
import torch.nn as nn
import torch.nn.functional as F
import source.stable_diffusion as sd
from source.stable_diffusion_labels import NegativeLabel
from itertools import chain
import torchvision.transforms.functional as TF

# === 核心工具函数 ===

class LearnableVolume(nn.Module):
    def __init__(self, size=64):
        super().__init__()
        self.size = size
        # 初始化一个随机的 3D 体素网格 (64x64x64)
        # 初始值设小一点，方便优化
        self.voxels = nn.Parameter(torch.randn(size, size, size) * 0.1)

    def forward(self):
        # 使用 Sigmoid 将值限制在 0-1 之间，代表“密度”或“存在物质的概率”
        return torch.sigmoid(self.voxels)

def get_projection(volume, axis):
    """
    计算体素在指定轴上的投影。
    axis=0: 从侧面看 (YZ平面)
    axis=1: 从正面看 (XZ平面)
    axis=2: 从顶上看 (XY平面)
    """
    # 1. 沿着视线方向求平均 (模拟X光或者是半透明物体叠加)
    # 也可以用 max，但在梯度下降中 mean 通常更平滑
    proj = torch.mean(volume, dim=axis)
    
    # 2. 归一化/增强对比度 (让图像更清晰，像墨水画)
    # 乘以一个系数让密度更实
    proj = proj * 3.0 
    proj = torch.tanh(proj)
    
    # 3. 因为体素只有 64x64，SD 需要 256x256 以上，所以需要上采样
    # 增加 batch 和 channel 维度: [1, 1, 64, 64]
    proj = proj.unsqueeze(0).unsqueeze(0)
    
    # 双线性插值放大到 256x256
    proj_upscaled = F.interpolate(proj, size=(256, 256), mode='bilinear', align_corners=False)
    
    # 去掉多余维度，变成 [3, 256, 256] (复制3份变成RGB)
    return proj_upscaled.squeeze(0).repeat(3, 1, 1)

print("✅ 3D 体素投影工具定义完成！现在你应该可以正常继续了。")

In [ ]:
# 初始化 GPU 和 模型
if 'model_sd' not in dir():
    print("正在加载 Stable Diffusion...")
    model_name = "CompVis/stable-diffusion-v1-4"
    gpu = rp.select_torch_device()
    model_sd = sd.StableDiffusion(gpu, model_name)
    device = model_sd.device
    print("模型加载完毕！")
else:
    print("模型已存在，跳过加载。")

In [ ]:
# === 🎮 3D 幻觉参数 ===

# 请在这里定义三个方向想看到什么
# 建议：物体形状差异大一些更有趣
PROMPT_X = "A pixel art of a red apple, white background"       # X轴投影
PROMPT_Y = "A pixel art of a blue butterfly, white background"    # Y轴投影
PROMPT_Z = "A pixel art of a green tree, white background"        # Z轴投影

# 负面提示词 (通用的)
negative_prompt = "blur, noise, text, letters, low quality, ugly, distortion, messy"

# === 初始化 ===
# 创建可学习的 3D 体素 (分辨率 64x64x64)
# 如果显存够大，可以尝试改大 size，比如 96 或 128
volume_model = LearnableVolume(size=64).to(device)

# 准备标签
label_x = NegativeLabel(PROMPT_X, negative_prompt)
label_y = NegativeLabel(PROMPT_Y, negative_prompt)
label_z = NegativeLabel(PROMPT_Z, negative_prompt)

# 优化器
# 学习率给大一点，因为是从随机噪声开始捏 3D 形状
optim = torch.optim.Adam(volume_model.parameters(), lr=0.01)

print(f"初始化完成。")
print(f"X轴目标: {PROMPT_X}")
print(f"Y轴目标: {PROMPT_Y}")
print(f"Z轴目标: {PROMPT_Z}")

In [ ]:
# === 修复说明 ===
# 主要修改了 loss 的计算部分，增加了 .mean() 以防止形状报错
# ================

NUM_ITER = 3000           # 训练步数
DISPLAY_INTERVAL = 200    # 显示频率

model_sd.max_step = 980
model_sd.min_step = 20

display_eta = rp.eta(NUM_ITER, title='Training 3D Illusion')

print("🚀 开始 3D 雕刻训练...")

try:
    for iter_num in range(NUM_ITER):
        display_eta(iter_num)
        
        # 1. 获取当前的 3D 体素数据
        vol = volume_model()
        
        # 2. 获取三个方向的投影图 (Projected Images)
        img_x = get_projection(vol, axis=0)
        img_y = get_projection(vol, axis=1)
        img_z = get_projection(vol, axis=2)
        
        # 3. 计算 Stable Diffusion Loss
        # 【关键修复】在后面加上 .mean() 确保它是一个标量数字
        loss_x = model_sd.train_step(
            label_x.embedding, img_x[None], noise_coef=0.1, guidance_scale=60
        ).mean()
        
        loss_y = model_sd.train_step(
            label_y.embedding, img_y[None], noise_coef=0.1, guidance_scale=60
        ).mean()
        
        loss_z = model_sd.train_step(
            label_z.embedding, img_z[None], noise_coef=0.1, guidance_scale=60
        ).mean()
        
        # 总 Loss
        total_loss = loss_x + loss_y + loss_z
        
        # (可选) 稀疏性正则化
        # 这里我们希望体素值远离 0.5 (即接近 0 或 1)
        # 既然我们是做 backward(total_loss - sparsity_loss) = minimize(total - sparsity)
        # = minimize(total) + maximize(sparsity)
        # maximize(|vol - 0.5|) 就是让值远离 0.5
        sparsity_loss = torch.mean(torch.abs(vol - 0.5)) * 0.1
        
        # 4. 反向传播
        # 注意：这里我们是在最小化 "总误差减去稀疏度"，等同于"最小化误差 并 最大化稀疏度"
        (total_loss - sparsity_loss).backward() 

        # --- C. 显示进度 ---
        with torch.no_grad():
            if iter_num % DISPLAY_INTERVAL == 0:
                from IPython.display import clear_output
                clear_output(wait=True)
                
                # 转为 Numpy 方便显示
                np_x = rp.as_numpy_image(img_x)
                np_y = rp.as_numpy_image(img_y)
                np_z = rp.as_numpy_image(img_z)
                
                print(f"Iteration {iter_num} / {NUM_ITER}")
                print(f"X轴 (侧视): {PROMPT_X}")
                print(f"Y轴 (正视): {PROMPT_Y}")
                print(f"Z轴 (顶视): {PROMPT_Z}")
                
                # 拼接显示
                combined = np.hstack([np_x, np_y, np_z])
                rp.display_image(combined)

        optim.step()
        optim.zero_grad()

except KeyboardInterrupt:
    print("用户手动停止训练。")

In [ ]:
print("==== 最终 3D 成果展示 ====")

with torch.no_grad():
    vol = volume_model()
    
    final_x = get_projection(vol, axis=0)
    final_y = get_projection(vol, axis=1)
    final_z = get_projection(vol, axis=2)
    
    print(f"1. X轴视角: {PROMPT_X}")
    rp.display_image(rp.as_numpy_image(final_x))
    
    print(f"2. Y轴视角: {PROMPT_Y}")
    rp.display_image(rp.as_numpy_image(final_y))
    
    print(f"3. Z轴视角: {PROMPT_Z}")
    rp.display_image(rp.as_numpy_image(final_z))

    # 保存结果
    rp.save_image(rp.as_numpy_image(final_x), "projection_x.png")
    rp.save_image(rp.as_numpy_image(final_y), "projection_y.png")
    rp.save_image(rp.as_numpy_image(final_z), "projection_z.png")
    print("\n✅ 图片已保存到左侧文件栏。")

In [ ]:
import numpy as np
from skimage import measure
try:
    from google.colab import files
except ImportError:
    pass # 如果不在 Colab 也没事

def export_volume_to_obj(vol_model, filename="my_illusion.obj", threshold=0.4):
    """
    将体素模型转换为 OBJ 网格文件
    threshold: 阈值 (0-1)。数值越小，物体越“胖”；数值越大，物体越“瘦”/破碎。
               推荐 0.3 到 0.5 之间尝试。
    """
    print(f"正在导出 {filename} (阈值: {threshold})...")
    
    # 1. 获取体素数据 (转为 numpy)
    # vol_model() 输出的是 sigmoid 后的 0-1 概率值
    with torch.no_grad():
        volume_data = vol_model().detach().cpu().numpy()
    
    # 2. 使用 Marching Cubes 算法生成网格
    # verts: 顶点坐标, faces: 面索引
    try:
        verts, faces, normals, values = measure.marching_cubes(volume_data, level=threshold)
    except ValueError:
        print("❌ 导出失败：模型太稀疏了，没有形成闭合表面。请尝试降低 threshold (例如 0.2)。")
        return

    # 3. 居中模型 (让它绕中心旋转，而不是绕角落)
    # 体素大小是 64x64x64，所以减去 32 把原点移到中心
    center_offset = np.array(volume_data.shape) / 2
    verts = verts - center_offset
    
    # 4. 写入 OBJ 文件
    with open(filename, 'w') as f:
        f.write(f"# 3D Illusion Export\n")
        f.write(f"o Illusion\n")
        
        # 写入顶点 (v x y z)
        # 注意：这里我们交换一下 y 和 z，适应大多数 3D 查看器的坐标系 (Y-up)
        for v in verts:
            f.write(f"v {v[0]} {v[2]} {v[1]}\n")
            
        # 写入面 (f v1 v2 v3)
        # OBJ 索引从 1 开始，所以要 +1
        for face in faces:
            f.write(f"f {face[0]+1} {face[1]+1} {face[2]+1}\n")
            
    print(f"✅ 成功导出！文件大小: {os.path.getsize(filename)/1024:.2f} KB")
    
    # 如果在 Colab，自动触发下载
    if 'google.colab' in str(get_ipython()):
        files.download(filename)

# === 执行导出 ===
# 你可以调整 threshold。如果导出的东西太像一团雾，就调高一点；如果像破碎的饼干，就调低一点。
export_volume_to_obj(volume_model, "illusion_model.obj", threshold=0.4)